<a href="https://colab.research.google.com/github/tekpinar/gromacscolab/blob/main/MD_Execution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#GROMACSCOLAB: MD Simulation Execution Notebook

🔷 Workflow Overview

You will:

*   Run minimization
*   Run NVT equilibration
*   Run NPT equilibration
*   Run production simulation





# You have three options on Gromacs version to use:

1.   Use prebuilt Gromacs: This is a Gromacs 2024.3 precompiled for Google Colab. It is the recommended option!
2.   Use system Gromacs: The installation is fast but the program is very slow!
3.   Compile Gromacs: It takes about 20 minutes to compile Gromacs but it may give you the best performance!



In [ ]:
whichGromacs = "Compile Gromacs" #@param ["Use prebuilt Gromacs", "Use system Gromacs",  "Compile Gromacs"]

# Install Gromacs
The next cell installs the GROMACS molecular dynamics package in the current environment and verifies the installation by printing the program version. Confirming the installed version is important for reproducibility, since simulation behavior and available features may vary between different software releases.

Recording the software version ensures that the molecular dynamics workflow can be replicated under the same computational conditions and allows consistent comparison of simulation results across different systems or computing platforms.

In [ ]:

if whichGromacs == "Use system Gromacs":
  !apt update -y >/dev/null
  !apt install -y gromacs >/dev/null
  !gmx --version



In [ ]:
if whichGromacs == "Compile Gromacs":
  # Check the GPU compute capability!
  !nvidia-smi --query-gpu=compute_cap --format=csv,noheader | tr -d '.'

  # If you don't see anything, use Runtime menu and select 'Change runtime type' to select a GPU (for example, T4)
  # Here, I am trying to compile Gromacs for utilizing all resources (CPU+GPU)
  # 1-Get the required libraries
  !apt-get update -q
  !apt-get install -y cmake build-essential libfftw3-dev

  # 2-Check Available Resources
  # Check GPU
  !nvidia-smi
  # Check CPU cores
  !nproc
  # Check RAM
  !free -h

  #3-Download & Extract GROMACS
  !wget ftp://ftp.gromacs.org/gromacs/gromacs-2024.3.tar.gz
  !tar xfz gromacs-2024.3.tar.gz
  %cd gromacs-2024.3

  #4-Configure with CMake
  !mkdir build && cd build && cmake .. \
    -DGMX_BUILD_OWN_FFTW=ON \
    -DREGRESSIONTEST_DOWNLOAD=OFF \
    -DGMX_MPI=OFF \
    -DGMX_GPU=CUDA \
    -DGMX_CUDA_TARGET_SM="75;80;86;89" \
    -DGMX_SIMD=AVX2_256 \
    -DGMX_FFT_LIBRARY=fftw3 \
    -DCMAKE_BUILD_TYPE=Release \
    -DGMX_OPENMP=ON \
    -DGMX_DOUBLE=OFF

  #5-Compile & Install
  !cd /content/gromacs-2024.3/build && make -j$(nproc) && make install

  #6-Source the Environment
  !source /usr/local/gromacs/bin/GMXRC
  # Or add to shell permanently:
  !echo "source /usr/local/gromacs/bin/GMXRC" >> ~/.bashrc

  #7-Verify Installation

  !GMX_PATH='/usr/local/gromacs/bin'
  !$GMX_PATH/gmx --version
  #!source /usr/bin/GMXRC
  !echo $GMX_PATH
  !echo $PATH

In [ ]:
# # This cell is just for suppressing the installation log

# # #5-Compile & Install
# # %%capture
# !cd /content/gromacs-2024.3/build && make -j$(nproc) && make install

# #6-Source the Environment
!source /usr/local/gromacs/bin/GMXRC
# Or add to shell permanently:
!echo "source /usr/local/gromacs/bin/GMXRC" >> ~/.bashrc

#7-Verify Installation
!gmx --version

In [ ]:
!source ~/.bashrc
# !alias gmx=$GMX_PATH/gmx
# !$GMX_PATH/gmx
!gmx

In [ ]:
# #%%bash
# #!source /usr/local/gromacs/bin/GMXRC
# #!/usr/local/gromacs/bin/gmx --version
# GMX_PATH='/usr/local/gromacs/bin'
# #!source /usr/bin/GMXRC
# !echo $GMX_PATH
# !echo $PATH

In [ ]:
!apt-get update -q
!apt-get install -y cmake build-essential libfftw3-dev

In [ ]:
if whichGromacs == "Use prebuilt Gromacs":
  # Get prebuilt Gromacs compiled for Google Colab from my zenodo repo.
  !wget https://zenodo.org/records/21284591/files/gromacs-2024.3-prebuilt-for-colab.tar.bz2
  !tar xjvf gromacs-2024.3-prebuilt-for-colab.tgz
  #!mv gromacs-2024.3-prebuilt-for-colab gromacs-2024.3
  !cd gromacs-2024.3/build && make install
  !GMX_PATH='/usr/local/gromacs/bin'
  !source /usr/local/gromacs/bin/GMXRC
  !echo $GMX_PATH
  !echo $PATH

In [ ]:
!gromacs-2024.3/build/bin/gmx

## File Upload for Simulation Input

This cell is used to upload files from the local computer. At this stage, the `.zip` archive generated and downloaded during the preparation step is typically re-uploaded, and its contents are made available in the working directory for subsequent simulation steps.

Since the notebook environment is temporary, user-provided data must be uploaded manually.

---


In [ ]:
from google.colab import files
files.upload()


This cell extracts the compressed molecular dynamics system archive and lists the available files in the working directory. This ensures that all required topology, structure, and parameter files are properly restored before continuing the simulation workflow.

In [ ]:
!unzip -o *.zip
!ls

# **Energy Minimization Step Setup**
This cell provides an interactive control to select the number of steps for the energy minimization stage. Energy minimization relaxes the initial structure by reducing steric clashes and unfavorable contacts introduced during system preparation. The chosen number of steps determines how thoroughly the potential energy of the system is reduced before equilibration.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

em_steps_widget = widgets.IntSlider(
    value=100000,
    min=5000,
    max=200000,
    step=5000,
    description='EM steps:',
    style={'description_width': 'initial'},
    continuous_update=False
)

display(em_steps_widget)

print("Select the number of energy minimization steps, then run the next cell.")


## **Generate minim.mdp File**
This cell generates the molecular dynamics parameter (MDP) file for the energy minimization stage. The steepest descent algorithm is used to iteratively reduce the potential energy of the system. The number of minimization steps is determined interactively by the user.

The Verlet cutoff scheme is applied for neighbor searching, and long-range electrostatic interactions are treated using the Particle Mesh Ewald (PME) method. Periodic boundary conditions are enabled in all spatial directions. These settings ensure that the system is physically consistent and numerically stable before proceeding to equilibration simulations.

In [ ]:
with open("minim.mdp", "w") as f:
    f.write(f"""
integrator  = steep
nsteps      = {em_steps_widget.value}
emtol       = 1000.0
emstep      = 0.01

cutoff-scheme = Verlet
nstlist       = 10
ns_type       = grid
coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0
pbc           = xyz
""")

print("minim.mdp created with", em_steps_widget.value, "steps")



## **Run Energy Minimization**
This cell performs the energy minimization of the solvated and ionized system. First, the prepared structure file is automatically located. The GROMACS preprocessor (grompp) combines the structure, topology, and minimization parameters to generate a run input file. Then, the mdrun command executes the minimization simulation.

During energy minimization, atomic positions are iteratively adjusted to reduce the potential energy of the system. This removes steric clashes and unfavorable contacts introduced during solvation and ion placement, producing a physically stable starting configuration for equilibration simulations

In [ ]:
import glob

str_files = glob.glob("*_solv_ions.pdb")

if len(str_files) == 0:
    raise FileNotFoundError("*_solv_ions.pdb file not found!")

structure_file = str_files[0]
print("Structure file to be used:", structure_file)

!$GMX_PATH/gmx grompp -f minim.mdp -c {structure_file} -p topol.top -o em.tpr -maxwarn 2
!$GMX_PATH/gmx mdrun -v -deffnm em



# **Set NVT Step Number**
This cell allows the user to define the number of steps for the NVT equilibration stage. In the NVT ensemble, the number of particles (N), system volume (V), and temperature (T) are kept constant. The purpose of this stage is to stabilize the temperature of the system after energy minimization and allow the solvent molecules to reorganize around the protein.

The selected number of steps determines how long the system is allowed to equilibrate thermally before moving to the pressure equilibration (NPT) stage.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

nvt_steps_widget = widgets.IntSlider(
    value=500000,
    min=10000,
    max=1000000,
    step=10000,
    description='NVT steps:',
    style={'description_width': 'initial'},
    continuous_update=False
)

display(nvt_steps_widget)

print("Select the number of NVT steps, then run the next cell.")



## **Set NVT Temperature**
This cell sets the target temperature for the NVT equilibration stage. During this stage, a thermostat regulates the system so that it reaches the desired thermal condition, which determines the average kinetic energy and molecular motion of the particles.
The default value of 298.15 K (≈25 °C) represents standard laboratory room temperature and is commonly used to compare simulations with experimental biochemical measurements. Choosing the temperature therefore defines the physical environment being modeled.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

nvt_temp_widget = widgets.FloatText(
    value=298.15,
    description='Temperature (K):',
    step=1.0,
    style={'description_width': 'initial'}
)

display(nvt_temp_widget)

print("Set the temperature, then run the next cell.")


## **Create nvt.mdp File**
This cell creates the parameter file for the NVT equilibration simulation. In the NVT ensemble, the number of particles, system volume, and temperature are kept constant while the system is integrated using molecular dynamics. A thermostat (V-rescale) controls the temperature so that it approaches the selected target value, and initial velocities are generated according to the Maxwell–Boltzmann distribution.

Position restraints are applied to the protein, allowing the solvent and ions to relax around it without large structural distortions. The default temperature (typically 298.15 K) corresponds to room temperature conditions, enabling a realistic thermal environment before pressure equilibration.

In [ ]:
with open("nvt.mdp", "w") as f:
    f.write(f"""
define       = -DPOSRES
integrator   = md
nsteps       = {nvt_steps_widget.value}
dt           = 0.002

cutoff-scheme = Verlet
nstlist       = 10
ns_type       = grid
pbc           = xyz

coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0

tcoupl     = V-rescale
tc_grps    = Protein Non-Protein
tau_t      = 0.1 0.1
ref_t      = {nvt_temp_widget.value} {nvt_temp_widget.value}

pcoupl     = no

gen_vel    = yes
gen_temp   = {nvt_temp_widget.value}
gen_seed   = -1

constraints = h-bonds
""")

print("nvt.mdp created")



## **Run NVT Equilibration**
This cell performs the NVT equilibration simulation. The preprocessor (grompp) prepares the run input file using the minimized structure, topology, and NVT parameters. The molecular dynamics run (mdrun) then integrates the equations of motion while maintaining constant particle number, volume, and temperature.

During this stage, the system is heated to the target temperature and solvent molecules reorganize around the restrained protein. The purpose of NVT equilibration is to achieve thermal stability before pressure equilibration (NPT) and production simulations.

In [ ]:
!$GMX_PATH/gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr -maxwarn 2
!$GMX_PATH/gmx mdrun -v -deffnm nvt -nb gpu



## **Set NPT Step Number**
This cell allows the user to define the number of steps for the NPT equilibration stage. In the NPT ensemble, the number of particles (N), pressure (P), and temperature (T) are kept constant. During this stage, the simulation box size is allowed to change so that the system density and pressure reach physically realistic values.

The selected number of steps determines how long the system is equilibrated under pressure control before starting the production simulation.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

npt_steps_widget = widgets.IntSlider(
    value=100000,
    min=10000,
    max=1000000,
    step=10000,
    description='NPT steps:',
    style={'description_width': 'initial'},
    continuous_update=False
)

display(npt_steps_widget)

print("Select the number of NPT steps, then run the next cell.")


## **Create NPT Equilibration File**
This cell sets the target pressure for the NPT equilibration stage. During this stage, a barostat adjusts the simulation box volume so that the system pressure approaches the desired value. The default pressure of 1 bar corresponds to standard atmospheric conditions and enables the system density to reach a realistic value.

Controlling the pressure is important because solvent density and intermolecular distances depend on it, which directly influences protein–solvent interactions before the production simulation.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

npt_pressure_widget = widgets.FloatText(
    value=1.0,
    description='Pressure (bar):',
    step=0.1,
    style={'description_width': 'initial'}
)

display(npt_pressure_widget)

print("Set the pressure, then run the next cell.")



#**Create npt.mdp**
This cell generates the parameter file for the NPT equilibration simulation. In the NPT ensemble, the number of particles, pressure, and temperature are controlled while the system evolves according to molecular dynamics. A thermostat maintains the target temperature, and a barostat adjusts the simulation box volume to reach the desired pressure.

Position restraints are still applied to the protein so that solvent and ions can equilibrate without large structural changes. As the box volume changes, the system density approaches a realistic value, preparing the system for the production simulation under stable thermodynamic conditions.

In [ ]:
with open("npt.mdp", "w") as f:
    f.write(f"""
define       = -DPOSRES
integrator   = md
nsteps       = {npt_steps_widget.value}
dt           = 0.002

cutoff-scheme = Verlet
nstlist       = 10
ns_type       = grid
pbc           = xyz

coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0

tcoupl     = V-rescale
tc_grps    = Protein Non-Protein
tau_t      = 0.1 0.1
ref_t      = {nvt_temp_widget.value} {nvt_temp_widget.value}

pcoupl           = Berendsen
pcoupltype       = isotropic
tau_p            = 2.0
ref_p            = {npt_pressure_widget.value}
compressibility  = 4.5e-5

constraints = h-bonds
continuation = yes
gen_vel      = no
""")

print("npt.mdp created")



## **Run NPT Equilibration**
This cell performs the NPT equilibration simulation. The GROMACS preprocessor prepares the run input file using the NVT-equilibrated structure and simulation parameters. The molecular dynamics run then continues while controlling temperature and pressure.

During this stage, the simulation box volume is allowed to change so that the system reaches the target pressure and realistic density. Solvent and ions further relax around the restrained protein, producing a stable thermodynamic state suitable for the production molecular dynamics simulation.

In [ ]:
!$GMX_PATH/gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 2
!$GMX_PATH/gmx mdrun -v -deffnm npt



## **Set MD (Production) Step Number**
This cell allows the user to define the number of steps for the production molecular dynamics (MD) simulation. During this stage, the equilibrated system evolves freely according to the equations of motion, and structural and dynamical properties of the protein can be analyzed over time.

The selected number of steps determines the total simulation time and therefore the extent of conformational sampling obtained from the trajectory.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

md_steps_widget = widgets.IntSlider(
    value=500000,
    min=10000,
    max=2000000,
    step=10000,
    description='MD steps:',
    style={'description_width': 'initial'},
    continuous_update=False
)

display(md_steps_widget)

print("Select the number of MD steps, then run the next cell.")



## **Create Production MD Configuration File**
This cell creates the parameter file for the production molecular dynamics simulation. After equilibration, the system evolves under constant temperature and pressure while atomic motions are integrated over time. Long-range electrostatic interactions are calculated using the Particle Mesh Ewald (PME) method, and the Parrinello–Rahman barostat maintains the target pressure.

In this stage, no position restraints are applied, allowing the protein to move naturally. The resulting trajectory contains the time-dependent structural and dynamical behavior of the protein and will be used for subsequent analysis.

In [ ]:
with open("run.mdp", "w") as f:
    f.write(f"""
integrator   = md
nsteps       = {md_steps_widget.value}
dt           = 0.002

cutoff-scheme = Verlet
nstlist       = 10
pbc           = xyz

coulombtype   = PME
rcoulomb      = 1.0
rvdw          = 1.0

tcoupl     = V-rescale
tc_grps    = Protein Non-Protein
tau_t      = 0.1 0.1
ref_t      = {nvt_temp_widget.value} {nvt_temp_widget.value}

pcoupl           = Parrinello-Rahman
pcoupltype       = isotropic
tau_p            = 2.0
ref_p            = {npt_pressure_widget.value}
compressibility  = 4.5e-5

constraints = h-bonds
continuation = yes
gen_vel      = no
""")

print("run.mdp created")


## **Run Production MD Simulation**
This cell starts the production molecular dynamics simulation. The preprocessor (grompp) prepares the run input file using the equilibrated NPT structure, topology, and simulation parameters. The mdrun command then integrates the equations of motion over time to generate the molecular trajectory.

In this stage, the system evolves without position restraints under controlled temperature and pressure. The resulting trajectory represents the time-dependent behavior of the protein in a solvated environment and provides the data used for structural and dynamical analyses.



In [ ]:
!$GMX_PATH/gmx grompp -f run.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr -maxwarn 2
# tpr file is sufficient for now! Try to run the command below
# if you are really sure. You can download the md.tpr file and
# run the command below on a good workstation or a computing center!

!$GMX_PATH/gmx mdrun -v -deffnm md



## **Create ZIP Archive of All Generated Files and Purpose**
This cell creates a ZIP archive containing all files produced during the GROMACS simulation and downloads it to the local computer. The archive includes structure files, trajectories, energy outputs, and log files required for post-simulation analysis.

These files will be used to evaluate the simulation results, such as monitoring potential energy, temperature, and pressure stability, as well as performing structural analyses (e.g., RMSD and conformational changes). Packaging all outputs into a single archive also allows convenient transfer, sharing, and reproducibility of the simulation.

In [ ]:
!tar czvf gromacs-2024.3-built.tgz gromacs-2024.3

In [ ]:
%cd /content/
!pwd
!tar cjvf gromacs-2024.3-prebuilt-for-colab.tar.bz2 gromacs-2024.3

In [ ]:
import zipfile
from google.colab import files
import os

zip_name = "gromacs_simulation_outputs.zip"

dirs_to_zip = ["minim", "equil_nvt", "equil_npt", "prod"]

with zipfile.ZipFile(zip_name, 'w') as zipf:
    for dir_name in dirs_to_zip:
        for foldername, subfolders, filenames in os.walk(dir_name):
            for filename in filenames:
                file_path = os.path.join(foldername, filename)
                zipf.write(file_path)

print(f"{zip_name} is ready to download.")

files.download(zip_name)

